# Exercise 1 — Prompt Chaining for a Customer Support AI

**Goal:** build a multi-step prompt chain that simulates a customer-service flow, where the output of each step is fed into the next.

**Tools:** Google Colab · Anthropic Python SDK (Claude) · Python `json`/`re`

**Chain design**

| Step | Stage | Input | Output |
|---|---|---|---|
| 1 | Triage / classify | raw ticket | JSON: category, urgency, sentiment, summary |
| 2 | Gather missing info | ticket + Step 1 JSON | JSON: missing_info, questions (≤3) |
| 2b | Customer simulator | Step 2 questions + persona | customer's answers |
| 3 | Propose resolution | ticket + Step 1 + Step 2 + 2b answers | customer reply + internal notes |
| 4 | Escalation rule | Step 1 JSON + Step 3 internal notes | JSON: escalate, team, reason |

Each stage has its own system prompt with a role, constraints, and a required output format, so the next stage can parse it reliably.

## Setup

Uses the Anthropic Python SDK. Add your key to Colab **Secrets** (🔑 icon in the left sidebar) as `ANTHROPIC_API_KEY`, or paste it when prompted.

In [1]:
!pip install -q anthropic

import os, json, re, textwrap
from anthropic import Anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

client = Anthropic()
MODEL = "claude-sonnet-5"   # any current Claude model works; "claude-haiku-4-5" is cheaper/faster

def get_text(resp) -> str:
    """Join the text blocks of a response (newer models may also return thinking blocks)."""
    return "".join(b.text for b in resp.content if b.type == "text").strip()

def ask(system: str, user: str, max_tokens: int = 4000) -> str:
    """Single-turn call: system prompt + one user message -> text."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
        thinking={"type": "disabled"},   # the prompts themselves carry the reasoning steps
    )
    return get_text(resp)

def parse_json(text: str):
    """Tolerant JSON parse: strips ```json fences if the model adds them."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    return json.loads(cleaned)

def show(title: str, body: str):
    print(f"\n{'='*70}\n{title}\n{'='*70}\n{body}")

print("Ready. Model:", MODEL)

Ready. Model: claude-sonnet-5


## Sample tickets

In [2]:
TICKETS = {
    "T-101": "Hi, I ordered running shoes (order #48213) 12 days ago and tracking still says 'label created'. "
             "I need them before my half-marathon this Saturday. Honestly this is really frustrating.",
    "T-102": "I was charged twice for my monthly membership — two $29.99 charges on the card ending 4417. "
             "Please fix this.",
    "T-103": "How do I change the email address on my account?",
}
for k, v in TICKETS.items():
    print(k, "→", v, "\n")

T-101 → Hi, I ordered running shoes (order #48213) 12 days ago and tracking still says 'label created'. I need them before my half-marathon this Saturday. Honestly this is really frustrating. 

T-102 → I was charged twice for my monthly membership — two $29.99 charges on the card ending 4417. Please fix this. 

T-103 → How do I change the email address on my account? 



## Step 1 — Triage / classification

### Iteration: v1 (before) → v2 (after)
My first classifier prompt was one line. The output was prose that varied in shape between tickets, so Step 2 couldn't parse it. v2 fixes the role, allowed values, and output format.

In [3]:
# ---- v1: first attempt (kept to show iteration) ----
TRIAGE_V1 = "Classify this customer message."
show("v1 output for T-101 (unstructured — hard to chain)", ask(TRIAGE_V1, TICKETS["T-101"]))


v1 output for T-101 (unstructured — hard to chain)
**Classification:**

- **Category:** Shipping/Delivery Issue — Order Not Moving (Stuck in Pre-Transit)
- **Subcategory:** Time-sensitive/urgent request
- **Sentiment:** Negative/Frustrated (but controlled, not abusive)
- **Priority:** High (deadline-driven — event this Saturday)

**Key details extracted:**
- Order #48213
- Item: Running shoes
- Order age: 12 days
- Tracking status: "Label created" only (no carrier movement/scan)
- Deadline: Needed before Saturday half-marathon
- Customer tone: Frustrated, but polite and factual

**Suggested handling:**
1. Treat as urgent — flag for expedited review given the hard deadline.
2. Contact carrier to check if package is truly unshipped or lost in transit.
3. Proactively offer solutions: reshipping with expedited shipping, refund, or store credit if it can't arrive in time.
4. Acknowledge frustration in response and apologize for the delay before presenting a solution.

Want me to draft a re

In [4]:
# ---- v2: role, allowed values, strict JSON ----
TRIAGE_SYSTEM = """You are the TRIAGE stage of a customer-support pipeline for an online sportswear retailer.
Classify the customer's ticket.

Return ONLY a JSON object with exactly these keys:
- "category": one of ["shipping", "billing", "account", "product_issue", "returns", "other"]
- "urgency": one of ["low", "medium", "high"]  (high = time-sensitive deadline, money lost, or safety)
- "sentiment": one of ["positive", "neutral", "negative"]
- "summary": a factual summary of the issue in 20 words or fewer

Rules: no prose, no markdown fences, no extra keys. Do not invent details not in the ticket."""

def step1_triage(ticket: str) -> dict:
    return parse_json(ask(TRIAGE_SYSTEM, f"Ticket:\n{ticket}", max_tokens=4000))

triage = {tid: step1_triage(t) for tid, t in TICKETS.items()}
for tid, t in triage.items():
    print(tid, json.dumps(t))

T-101 {"category": "shipping", "urgency": "high", "sentiment": "negative", "summary": "Order #48213 stuck at 'label created' for 12 days; customer needs shoes before Saturday half-marathon."}
T-102 {"category": "billing", "urgency": "high", "sentiment": "negative", "summary": "Customer charged twice for monthly membership, two $29.99 charges on card ending 4417, requests correction."}
T-103 {"category": "account", "urgency": "low", "sentiment": "neutral", "summary": "Customer asks how to change the email address associated with their account."}


## Step 2 — Gather missing information
Uses the Step 1 JSON to decide what an agent still needs. Constraints: ≤3 questions, only ask for things not already in the ticket, never ask for full card numbers or passwords.

In [5]:
GATHER_SYSTEM = """You are the INFORMATION-GATHERING stage of a customer-support pipeline.
You receive the customer's ticket and the triage result (category, urgency, sentiment).

Task: decide what information a support agent still needs before the issue can be resolved,
and write the questions to ask the customer.

Rules:
- Ask at most 3 questions. If nothing is missing, return an empty list.
- Only ask for information that is NOT already in the ticket.
- Questions must be specific to the category (e.g. shipping -> order number, shipping address on file).
- NEVER ask for full card numbers, passwords, or one-time codes.
- Tone: warm and concise, one sentence per question.

Return ONLY a JSON object: {"missing_info": [<short labels>], "questions": [<questions to the customer>]}"""

def step2_gather(ticket: str, triage_json: dict) -> dict:
    user = f"Ticket:\n{ticket}\n\nTriage result:\n{json.dumps(triage_json)}"
    return parse_json(ask(GATHER_SYSTEM, user, max_tokens=4000))

gather = {tid: step2_gather(TICKETS[tid], triage[tid]) for tid in TICKETS}
for tid, g in gather.items():
    print(tid, json.dumps(g, indent=2))

T-101 {
  "missing_info": [
    "shipping address on file",
    "preferred resolution"
  ],
  "questions": [
    "Could you please confirm the shipping address you used for this order so we can verify it's correct?",
    "Since your race is Saturday, would you prefer we expedite a replacement shipment or process a refund if we can't guarantee delivery in time?"
  ]
}
T-102 {
  "missing_info": [
    "account/membership identifier",
    "dates of the duplicate charges",
    "preferred resolution"
  ],
  "questions": [
    "Could you please provide the email address or account ID associated with your membership?",
    "What were the dates of the two $29.99 charges you noticed?",
    "Would you prefer a refund for the duplicate charge or a credit toward your next billing cycle?"
  ]
}
T-103 {
  "missing_info": [
    "current account email or username",
    "new email address to update to"
  ],
  "questions": [
    "Could you confirm the email address or username currently on your account?"

## Step 2b — Simulated customer answers
A small "customer simulator" prompt so the chain runs end-to-end without a human. It only answers the questions from Step 2 (prior output → input).

In [6]:
CUSTOMER_SYSTEM = """You are role-playing the CUSTOMER who wrote the ticket below.
Answer the support agent's questions briefly and realistically, staying consistent with the ticket.
Invent plausible details only where the ticket is silent (e.g. an address or a date).
Return ONLY a JSON object mapping each question (verbatim) to your answer. If there are no questions, return {}."""

def step2b_customer(ticket: str, questions: list) -> dict:
    if not questions:
        return {}
    user = f"Your original ticket:\n{ticket}\n\nAgent's questions:\n{json.dumps(questions)}"
    return parse_json(ask(CUSTOMER_SYSTEM, user, max_tokens=4000))

answers = {tid: step2b_customer(TICKETS[tid], gather[tid]["questions"]) for tid in TICKETS}
for tid, a in answers.items():
    print(tid, json.dumps(a, indent=2))

T-101 {
  "Could you please confirm the shipping address you used for this order so we can verify it's correct?": "Yes, it's 214 Maple Street, Apt 3B, Columbus, OH 43215.",
  "Since your race is Saturday, would you prefer we expedite a replacement shipment or process a refund if we can't guarantee delivery in time?": "I'd prefer an expedited replacement if you can guarantee it arrives by Friday, otherwise just refund me so I can buy locally."
}
T-102 {
  "Could you please provide the email address or account ID associated with your membership?": "It's jmartinez82@gmail.com",
  "What were the dates of the two $29.99 charges you noticed?": "They were both posted on the same day, March 3rd, back to back.",
  "Would you prefer a refund for the duplicate charge or a credit toward your next billing cycle?": "I'd prefer a refund back to my card, please."
}
T-103 {
  "Could you confirm the email address or username currently on your account?": "The current email on my account is jsmith82@gmail

## Step 3 — Propose a resolution
Receives everything so far (ticket, triage, missing info, customer answers). Produces a customer-facing reply **and** internal notes, in tagged sections so Step 4 can pull out just the notes.

In [7]:
RESOLVE_SYSTEM = """You are the RESOLUTION stage of a customer-support pipeline for an online sportswear retailer.
You receive the ticket, the triage result, and the customer's answers to clarifying questions.

Company policy you must follow:
- Shipping: if tracking has not moved for 7+ days we can reship or refund; expedited reship is free if a stated deadline is within 5 days.
- Billing: duplicate charges are refunded within 3-5 business days; do not promise same-day refunds.
- Account: email changes are done by the customer under Settings > Profile; agents never change it for them.

Write two sections, using exactly these tags:
<reply>
A customer-facing message: at most 120 words, first sentence acknowledges the specific issue,
then concrete numbered next steps, plain language, no jargon, no emojis. Sign off as "Sam, Customer Care".
</reply>
<internal_notes>
2-4 short bullet points for the agent: what was done, anything unresolved, any refund amount, any risk.
</internal_notes>"""

def step3_resolve(tid: str) -> dict:
    user = (f"Ticket:\n{TICKETS[tid]}\n\nTriage:\n{json.dumps(triage[tid])}\n\n"
            f"Missing info requested:\n{json.dumps(gather[tid]['missing_info'])}\n\n"
            f"Customer's answers:\n{json.dumps(answers[tid])}")
    text = ask(RESOLVE_SYSTEM, user, max_tokens=4000)
    reply = re.search(r"<reply>(.*?)</reply>", text, re.S).group(1).strip()
    notes = re.search(r"<internal_notes>(.*?)</internal_notes>", text, re.S).group(1).strip()
    return {"reply": reply, "internal_notes": notes}

resolution = {tid: step3_resolve(tid) for tid in TICKETS}
for tid, r in resolution.items():
    show(f"{tid} — reply", r["reply"])
    print("\nInternal notes:\n" + r["internal_notes"])


T-101 — reply
I'm sorry about this, waiting 12 days with no movement on your shoes is frustrating, especially with your race so close. Here's what I'm doing:

1. I've confirmed your address (214 Maple Street, Apt 3B, Columbus, OH 43215) is correct.
2. Since tracking hasn't moved in over 7 days, I'm sending a free expedited replacement today.
3. If our carrier cannot guarantee delivery by Friday, I'll cancel the reship and issue a full refund instead so you can buy locally in time.
4. You'll get a confirmation email within the next few hours with tracking or refund details.

Good luck at your half-marathon this Saturday!

Sam, Customer Care

Internal notes:
- Confirmed address matches order; no discrepancy found.
- Original package stuck at "label created" for 12 days — qualifies for free expedited reship per policy.
- Action: initiate expedited reship now; verify carrier can commit to Friday delivery before customer's deadline.
- Fallback: if expedited delivery cannot be guaranteed, c

## Step 4 — Escalation rule
Deterministic Python rules **plus** an LLM judgment on the internal notes. The rule layer guarantees the obvious cases; the LLM catches things the rules miss (e.g. a threat to dispute the charge).

In [8]:
ESCALATE_SYSTEM = """You are the ESCALATION GATE of a customer-support pipeline.
Decide whether a human specialist must take over, based on the triage result and the agent's internal notes.

Escalate (true) if ANY of these hold:
- urgency is "high" AND the issue is not fully resolved in the notes
- a refund or credit over $100 is involved
- the customer mentions legal action, a chargeback/dispute, injury, or safety
- sentiment is "negative" and the notes list anything unresolved
Otherwise do not escalate.

Return ONLY a JSON object: {"escalate": true|false, "team": "shipping"|"billing"|"account"|"trust_safety"|"none", "reason": "<one sentence>"}"""

def step4_escalate(tid: str) -> dict:
    t, notes = triage[tid], resolution[tid]["internal_notes"]
    llm = parse_json(ask(ESCALATE_SYSTEM, f"Triage:\n{json.dumps(t)}\n\nInternal notes:\n{notes}", max_tokens=4000))
    # Hard rule overlay: never let the model downgrade these.
    if t["urgency"] == "high" and t["sentiment"] == "negative":
        llm["escalate"] = True
        llm["team"] = llm.get("team") if llm.get("team") != "none" else t["category"]
        llm["reason"] += " [rule: high urgency + negative sentiment]"
    return llm

for tid in TICKETS:
    print(tid, json.dumps(step4_escalate(tid)))

T-101 {"escalate": true, "team": "shipping", "reason": "High urgency issue with a tight deadline remains unresolved pending carrier confirmation and a possible unquantified refund, requiring human follow-up. [rule: high urgency + negative sentiment]"}


T-102 {"escalate": true, "team": "billing", "reason": "High urgency issue with unresolved duplicate charge refund still pending processing. [rule: high urgency + negative sentiment]"}


T-103 {"escalate": false, "team": "none", "reason": "Low urgency, neutral sentiment, and the customer was successfully directed to self-service steps with no unresolved issues."}


## Full chain, end to end
One function that runs all stages for a new ticket and prints every intermediate output — this is the final successful run.

In [9]:
def run_chain(ticket: str):
    t  = step1_triage(ticket)
    g  = step2_gather(ticket, t)
    a  = step2b_customer(ticket, g["questions"])
    user = (f"Ticket:\n{ticket}\n\nTriage:\n{json.dumps(t)}\n\nMissing info requested:\n"
            f"{json.dumps(g['missing_info'])}\n\nCustomer's answers:\n{json.dumps(a)}")
    text = ask(RESOLVE_SYSTEM, user, max_tokens=4000)
    reply = re.search(r"<reply>(.*?)</reply>", text, re.S).group(1).strip()
    notes = re.search(r"<internal_notes>(.*?)</internal_notes>", text, re.S).group(1).strip()
    e = parse_json(ask(ESCALATE_SYSTEM, f"Triage:\n{json.dumps(t)}\n\nInternal notes:\n{notes}", max_tokens=4000))

    show("TICKET", ticket)
    show("STEP 1 · triage", json.dumps(t, indent=2))
    show("STEP 2 · missing info / questions", json.dumps(g, indent=2))
    show("STEP 2b · simulated customer answers", json.dumps(a, indent=2))
    show("STEP 3 · reply to customer", reply)
    show("STEP 3 · internal notes", notes)
    show("STEP 4 · escalation decision", json.dumps(e, indent=2))

run_chain("My new water bottle arrived with a cracked lid and leaked all over my gym bag. "
          "Order #51877. I'd like a replacement, and if this keeps happening I'll dispute the charge.")


TICKET
My new water bottle arrived with a cracked lid and leaked all over my gym bag. Order #51877. I'd like a replacement, and if this keeps happening I'll dispute the charge.

STEP 1 · triage
{
  "category": "product_issue",
  "urgency": "medium",
  "sentiment": "negative",
  "summary": "Water bottle arrived with cracked lid, leaked in gym bag; customer wants replacement for order #51877."
}

STEP 2 · missing info / questions
{
  "missing_info": [
    "photo of damaged lid",
    "shipping address confirmation"
  ],
  "questions": [
    "Could you share a photo of the cracked lid so we can process your replacement quickly?",
    "Can you confirm the shipping address we should send the replacement to?"
  ]
}

STEP 2b · simulated customer answers
{
  "Could you share a photo of the cracked lid so we can process your replacement quickly?": "Yes, I can take one and send it over today.",
  "Can you confirm the shipping address we should send the replacement to?": "Sure, it's the same as t

## Notes on testing & iteration
- **Step 1 v1 → v2:** the one-line v1 prompt (output above) returned a long markdown write-up with its own category names, a "priority" field, handling suggestions and a follow-up question — impossible to feed into Step 2. v2 (fixed enums + JSON only) returned clean, parseable JSON for all three tickets and the end-to-end ticket.
- **Engineering fixes while testing:** the model returns a thinking block before the text, so reading `content[0].text` crashed; `ask()` now joins only the text blocks. Thinking was also disabled so token budgets stay predictable, since the reasoning steps live in the prompts themselves.
- **Step 3:** the policy block in the system prompt kept replies inside policy — e.g. the billing reply quotes the 3–5 business day refund window, and the email-change reply sends the customer to Settings > Profile instead of an agent changing it.
- **Step 4:** the Python rule overlay fired on T-101 and T-102 (high urgency + negative) — visible as `[rule: ...]` in the reason. On the end-to-end ticket the LLM layer caught the chargeback threat and routed it to trust_safety, which the fixed rules alone would not have done.